## Structured Streaming

What is a stream?

A stream is an unbounded dataset, which has no theoretical beginning or ending.  This is unlike a batch which has a known size and processing time.  Streams can be messages or files processed in real-time as they arrive.

![](/Volumes/workspace/pyspark_learning/raw_files/images/unbounded_table.png)

### Compute Restrictions
To use trigger-based intervals, one needs to use an all-purpose compute.  Serverless compute does not support this trigger so I cannot do the example lesson in Databricks Free account

Structured Streaming

- is built on dataframe/dataset APIs
- contains event-time processing
- simplified API with SQL-like operations
- handles late and out of order data
- handled in small 'microbatched' chunks

Key Features of Structured Streaming
- Event-time processing
- Watermark support for late data
- End-to-end exactly once guarantees

In [0]:
# Watermark support in Spark Structured Streaming allows handling late-arriving data.
# It defines a threshold of how late data can be and still be processed.
# Data older than the watermark is considered too late and is dropped.

# Example: Setting watermark on a streaming DataFrame
streaming_df = spark.readStream.format("delta").load("/path/to/stream")
watermarked_df = streaming_df.withWatermark("event_time", "10 minutes")

Autoloader (cloud_files) is a Databricks source for high-performance cloud storage ingestion with auto schema handling

Read Stream Example
DataStreamReader creates streaming DataFrames
NOTE:  `readStream` method

```
df = spark.readStream \
    .format("kafka") \
    .option("kafka.boostrap.servers", "host:port") \
    .option("subscribe", "topic1") \
    .load()
```

After the stream is read into a dataframe, one can perform transformations as they normally would

Write stream example with DataStreamWriter

```
query = df.writeStream \
  .format("kafka") \
  .outputMode("append") \
  # .. more options ...
.start()
```

Triggers

- Default Trigger (as soon as possible).  This processes new data as soon as the previous micro-batch completes:  ``` df.writeStream.Start()```

- Fixed Interval Trigger.   Process data at specified time intervals, which is useful for controling resource usages/costs ``` df.writeStream.trigger(processingTime = '2 minuties').start() ```

- Available Now Trigger.  Process available data and then stops, won't wait for more new data to arrive ``` df.writeStream.trigger(availableNow=True).start() ```

Output Modes
- append (default): only adds new records to the sink
- update: modifies existing records and adds new ones. Only outputs records which changed since last trigger
- complete: writes entire result table to sink each time

Streaming Stateful vs Stateless

**Stateless**
- Process each record independently
- No memory of previous records
- Examples:  _select_, _filter_

**Stateful**
- Maintains information across batches
- Require a checkpoint location
- Examples: _groupBy_, _join_, _dropDuplicates_
- Window operations are stateful

Checkpoints

- Maintain state across batches
- Recover state in-case of failures
- Handle replay of data w/o duplicating results

_RocksDB_ is the backend state managing DB
A checkpoint directory is used for maintaining the state metadata

Streaming Joins

- All join types are supported except full and cross
- Streaming dataframes may be joined to other streaming dataframes or to static dataframes
- joins require maintaining state (increases memory usage)

Streaming Aggregations

Streaming datasets are unbound so aggregations are done in _windows_
- Tumbling window:  Fixed non-overlapping intervales (example: count events every 5 mins)
- Sliding windows:  Overlapping windows where an event can belong to multiple windows
- Session windows:  Dynamically sized windows based on user activity, (gaps or session timeouts)

### Autoloader
- Reads streams
- Incrementally processes data files as they arrive
- Provides a structured streaming source called `cloudFiles`
- Given an imput directory and path on the cloudf file storage, the `cloudFiles` source automatically processes new files as they arrive
- manages schema drift
- NOTE: all file types EXCEPT Delta is available to autoloader, you dont point autoloader to a delta table directly.  Its designed for raw-file ingestion

Example:
NOTE: format is always 'cloudFiles', then the raw data format is an option that is passed to the options method
The example below uses csv, but json, parquet, text, are also valid

Remember that `cloudFiles` is how we leverage the autoloader


Also, since the example below is csv, one must pass in a location where the autoloader can save the inferred schema information.  It needs this location it can save the work it already did inferring the current scheam and so it may track future schema changes.   This location is mandatory for csv and JSON files.


This is a transformation step only which is lazy evaluated
```
df = spark.readStream.\
    format("cloudFiles").\
    option("cloudFiles.format", "csv").\
    option("cloudFiles.schemaLoaction, "<path to schema file storage>").\
    load("<raw data source path>")
```


The stream starts when I issue an _action_ like below
The stream will run in the background until terminated, hence serverless compute not used
```
df.display()
```

### Micro-batch size
Streaming while seemingly continuous actually processes data in 'micr-batches'.

- default files processed at one time is 1000, but can be configured, `cloudFiles.maxFilesPerTrigger`
- can also limit by bytes, `cloudFilesMaxBytesPerTrigger`

These options can be used concurrently


The example below sets a max number of files trigger
```
df = spark.readStream.\
    format("cloudFiles").\
    option("cloudFiles.format", "csv").\
    option("cloudFiles.schemaLoaction, "<path to schema file storage>").\
    option("cloudFiles.maxFilesPerTrigger, "100").\
    load("<raw data source path>")
  ```

The example below sets a max bytes trigger of 10 GB
  ```
df = spark.readStream.\
    format("cloudFiles").\
    option("cloudFiles.format", "csv").\
    option("cloudFiles.schemaLoaction, "<path to schema file storage>").\
    option("cloudFiles.maxBytesPerTrigger, "10g").\
    load("<raw data source path>")
  ```

### Schema Inference and Evolution

- Schema inference is automatically enabled by simply adding the schema location to the readStream (required for csv and json)
- The autoloader will scan the first _50gb or 1000_ _files_, whichever is first and then infer a schema.  That schema is stored at the specified schema location path in a directory call `_schema`
- The stored schema is used to track future schema changes over time
- Autoloader can adapt to changes in the schema automatically
  - The stream will fail and throw a `UnkownFieldException` and the stream will stop BUT...
  - If a schema was not explictly provided, and is left to inference by autoloader, the next time the stream runs, it will perform an `addNewColumns` adding the new column into the scheam, processing the data
- Evolving the schema is defauly behavor but it can be overridden by setting the `cloudFiles.schemaEvolutionMode`
  - `rescue`:   Schema is never evovled and the stream does not fail due to schema changes.  All new columns are recorded in a 'rescued data column'
  - `failOnNewColumns`:  This is for explicitly supplied schemas.  Stream fails and does not restart unless the schema suppled upated or the offending data is removed
  - `none`:   Stream does not fail.  Does not evolve the schema, new columns are ignored and data is not rescued.


  Example which sets the schema evolution mode (if not default)...
  ```
  df = spark.readStream.\
  format("cloudFiles").\
  option("cloudFiles.format", "csv").\
  option("cloudFiles.schemaLoaction, "<path to schema file storage>").\
  option("cloudFiles.maxBytesPerTrigger, "10g").\
  option("cloudFiles.SchemaEvolutionMode", "none").\
  load("<raw data source path>")


  ```

### Watermark

Watermarks are used to control how long to wait for data to process.

Declaring a watermark.  The following example applies a 10 minute watermark to a windowed count.

- 10 minute watermark
- 5 minute tumbling window

```
from pyspark.sql.functions import window
(
  df.withWatermark("event_time", "10 minutes")
  .groupBy(window("event_time", "5 minutes"), "id")
  .count()
)  
  

```

### Writing Data from Streams

- Checkpoints are required for data writing.  They provide guarantees for structured streaming workloads.
- Checkpoints track the information which identifies the query including state information and and processed records.
- Allows for tbe ability of the stream to 'know' which files have been processed and which have not, so files are not processed twice.
- default write trigger intervals are 500ms, creating microbatches.  Specifying a trigger interval can reduce costs and is recommended.


No trigger
```
df.writeStream.\
 format('parquet').\
 outputMode('append').\
 option('checkpointLocation', <path to checkpoint location>).\
 option('path', <sink path>).\
 start()
```


Time trigger
```
df.writeStream.\
 format('parquet').\
 outputMode('append').\
 option('checkpointLocation', <path to checkpoint location>).\
 trigger(processingTime='10 seconds').\
 option('path', <sink path>).\
 start()
```


Continuous trigger (super low-latency trigger)
```
df.writeStream.\
 format('parquet').\
 outputMode('append').\
 option('checkpointLocation', <path to checkpoint location>).\
 trigger(continuous='1 second').\
 option('path', <sink path>).\
 start()
```


Available Now trigger (serverless only supports this trigger)
```
df.writeStream.\
 format('parquet').\
 outputMode('append').\
 option('checkpointLocation', <path to checkpoint location>).\
 trigger(availableNow=True).\
 option('path', <sink path>).\
 start()
```

### Writing to detla tables directly

```
df.writeStream.\
 outputMode('append').\
 option('checkpointLocation', <path to checkpoint location>).\
 toTable(catalog.schema.table)
```